# 10 — Evaluation

In [1]:
import sys, json
sys.path.insert(0, "..")
from src import config, data_loader, preprocessing, semantic_search, ranking, rag, evaluation

config.ensure_directories()
report = {}


## Dataset evaluation

In [2]:
df = data_loader.load_final_dataset()
dataset_report = preprocessing.validate_dataset(df)
report["dataset"] = dataset_report
dataset_report


2026-09-01 23:15:09,345 | INFO     | src.data_loader | Loaded final dataset: 583 rows from E:\Job Base Programe\ResearchMind\data\processed\research_papers_final.csv


{'num_rows': 583,
 'num_columns': 26,
 'columns_present': {'paper_id': 'paper_id',
  'title': 'title',
  'abstract': 'abstract',
  'authors': 'authors',
  'year': 'year',
  'venue': 'venue',
  'url': 'url',
  'citation_count': 'citation_count',
  'reference_count': 'reference_count',
  'publication_date': 'publication_date',
  'fields_of_study': 'fields_of_study',
  'influential_citation_count': 'influential_citation_count',
  'tldr': 'tldr',
  'research_topic': 'research_topic'},
 'columns_missing': [],
 'has_search_text': True,
 'duplicate_titles': 0}

## Semantic search evaluation

No manually labeled relevant_ids are supplied in this template run, so precision@k/recall@k/MRR are correctly reported as not computable. To enable them, populate `labeled_queries` with real relevance judgments.

In [3]:
labeled_queries = [
    # Example structure once labels exist:
    # {"query": "transformers in healthcare", "relevant_ids": ["<paper_id_1>", "<paper_id_2>"]},
]

retrieval_report = evaluation.evaluate_retrieval(
    labeled_queries,
    search_fn=lambda q, k: semantic_search.search_papers(q, top_k=k),
)
report["retrieval"] = retrieval_report
retrieval_report


{'num_queries': 0,
 'num_labeled_queries': 0,
 'precision_at_k': None,
 'recall_at_k': None,
 'mrr': None,
 'note': 'No manually labeled relevant_ids were supplied for any query, so precision@k, recall@k, and MRR cannot be reliably calculated.'}

## Similarity-score distribution (unsupervised diagnostic)

In [4]:
sample_results = semantic_search.search_papers("artificial intelligence in medicine", top_k=20)
scores = [r["similarity_score"] for r in sample_results]
sim_dist = evaluation.similarity_distribution(scores)
report["similarity_distribution"] = sim_dist
sim_dist


2026-09-01 23:15:10,350 | INFO     | src.vector_store | Loaded faiss-backed index with 583 vectors from E:\Job Base Programe\ResearchMind\vector_db\faiss_index\index.faiss
2026-09-01 23:16:13,680 | INFO     | src.embeddings | Loading sentence-transformers model 'all-mpnet-base-v2' (this may download weights on first use)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

{'count': 20,
 'mean': 0.60024,
 'std': 0.046615924317769376,
 'min': 0.5549,
 'max': 0.7295,
 'p50': 0.58135,
 'p90': 0.6678000000000001}

## Ranking evaluation

In [5]:
ranked = ranking.compute_scores(sample_results)
ranking_report = evaluation.evaluate_ranking(ranked)
report["ranking"] = ranking_report
ranking_report


{'num_papers': 20,
 'score_distribution': {'count': 20,
  'mean': 0.673135,
  'std': 0.039527765874129546,
  'min': 0.6186,
  'max': 0.7711,
  'p50': 0.6637,
  'p90': 0.7341300000000001},
 'is_correctly_sorted_desc': True}

## RAG evaluation

In [6]:
rag_result = rag.generate_answer("What AI methods are commonly applied in medicine?")
rag_report = evaluation.evaluate_rag(rag_result)
report["rag"] = rag_report
rag_report


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-09-01 23:16:25,629 | WARNING  | src.rag | LLM call failed, using extractive fallback: LLM_PROVIDER=anthropic but the `anthropic` package is not installed. Fix: pip install anthropic.


{'num_sources': 5,
 'context_relevance': True,
 'citation_correctness': True,
 'hallucination_rate': None,
 'retrieval_relevance': None,
 'note': 'hallucination_rate and retrieval_relevance require manual human review or labeled ground truth and are not fabricated here.'}

## Save full evaluation report

In [7]:
eval_path = config.EVALUATIONS_DIR / "evaluation_report.json"
with open(eval_path, "w") as f:
    json.dump(report, f, indent=2, default=str)
print("Saved evaluation report to", eval_path)


Saved evaluation report to E:\Job Base Programe\ResearchMind\outputs\evaluations\evaluation_report.json
